<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/7_eleven_allonline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## To DO
'''
1. Change column name
2. using polars
3. Add column : Date, retailer
4. Extract Brand from Thai dict
5. import function re_evaluate_price from other source
'''

In [7]:
%%capture
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install selenium webdriver-manager beautifulsoup4 pandas

In [11]:
from google.colab import data_table
data_table.enable_dataframe_formatter()


In [28]:
# from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
# import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-10


In [22]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd

# 1. Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

print("Starting browser...")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# Base URL without the "p=" parameter
base_url = "https://allonline.7eleven.co.th/supermarket/household-items/laundry/liquid-detergent/?filter.BRAND=Fineline&filter.BRAND=Hygiene&filter.BRAND=%E0%B9%80%E0%B8%9B%E0%B8%B2&filter.BRAND=%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84&filter.PRICE=49-470&filter.initial_from_PRICE=49&filter.initial_to_PRICE=470&landing=true&pageSize=90&sortBy=si&view=0"

data = []
p_index = 0 # THE FIX: Start counting at 0!
previous_page_names = []

while True:
    # We display (p_index + 1) in the print statement just so it reads nicely for us as "Page 1", "Page 2"
    print(f"\nScraping Page {p_index + 1} (URL parameter p={p_index})...")

    # Inject the 0-based parameter
    current_page_url = base_url + f"&p={p_index}"
    driver.get(current_page_url)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "item-description-cls-mobile"))
        )
    except Exception as e:
        print("No products found on this page. Reached the end!")
        break

    soup = BeautifulSoup(driver.page_source, "html.parser")
    product_titles = soup.find_all("div", class_="item-description-cls-mobile")

    if not product_titles:
        print("Page is empty. Reached the end!")
        break

    current_page_names = [title.text.strip() for title in product_titles]

    if current_page_names == previous_page_names:
        print("Duplicate items detected! We have reached the final page.")
        break

    previous_page_names = current_page_names.copy()

    # Parse Data
    for title_elem in product_titles:
        name = title_elem.text.strip()
        parent = title_elem.parent
        current_price_elem = None

        levels_climbed = 0
        while parent and parent.name != 'body' and levels_climbed < 5:
            current_price_elem = parent.find("strong")
            if current_price_elem:
                break
            parent = parent.parent
            levels_climbed += 1

        current_price = current_price_elem.text.strip() if current_price_elem else None
        original_price_elem = parent.find("s") if parent else None
        original_price = original_price_elem.text.strip() if original_price_elem else None

        data.append({
            "Product Name": name,
            "Current Price": current_price,
            "Original Price": original_price
        })

    print(f"Successfully grabbed {len(product_titles)} items from Page {p_index + 1}.")

    # Increase the index for the next loop (0 becomes 1, 1 becomes 2)
    p_index += 1

driver.quit()

# Clean Data
df = pd.DataFrame(data)

if not df.empty:
    df['Current Price'] = df['Current Price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
    df['Original Price'] = df['Original Price'].astype(str).str.replace(r'[^\d.]', '', regex=True)

    df['Current Price'] = pd.to_numeric(df['Current Price'])
    df['Original Price'] = pd.to_numeric(df['Original Price'])

    # Drop duplicates
    initial_count = len(df)
    df = df.drop_duplicates(subset=['Product Name'], keep='first').reset_index(drop=True)

    print("\n--- Final Scraping Results ---")
    print(df.tail(10))
    print(f"\nFinal clean item count: {len(df)}")

    filename = 'scoped_detergents.csv'
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\nData successfully saved to {filename}!")
else:
    print("No data was scraped.")

Starting browser...

Scraping Page 1 (URL parameter p=0)...
Successfully grabbed 90 items from Page 1.

Scraping Page 2 (URL parameter p=1)...
Successfully grabbed 13 items from Page 2.

Scraping Page 3 (URL parameter p=2)...
Duplicate items detected! We have reached the final page.

--- Final Scraping Results ---
                                          Product Name  Current Price  \
93   ไฟน์ไลน์ พลัส ผลิตภัณฑ์ซักผ้าชนิดน้ำ สูตรซีเคร...             69   
94      น้ำยาซักผ้าแอทแทค คลีน แอดวานซ์สูตรน้ำ 650 มล.             95   
95    น้ำยาซักผ้าแอทแทค เลดี้ อิลิแกนท์สูตรน้ำ 600 มล.             95   
96           แอทแทค น้ำยาซักผ้า บลู เอเมอรัลด์ 600 มล.             95   
97                เปา วินวอช น้ำยาซักผ้า สีแดง 620 มล.             99   
98   เปา วินวอชลิควิดไวท์ฟลอรั่ลถุงเติม น้ำยาซักผ้า...             99   
99         แอทแทค น้ำยาซักผ้า ไลฟ์ลี่ บลูมมิ่ง 700 มล.             95   
100          แอทแทค น้ำยาซักผ้า ลัคชูรี่ โกลว์ 600 มล.             95   
101  เปา วินวอชลิควิดเซ็นซว

In [23]:
df

,Product Name,Current Price,Original Price
0,ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ (แกลลอ...,238,271.0
1,ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ 1250 มล.,179,NaN
2,ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข้น 3in1 500 มล. ...,150,195.0
3,ไฟน์ไลน์ น้ำยาซักผ้า โปรคลีน ชมพู 550 มล. (1 แ...,150,195.0
4,ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข้น สำหรับซักกลาง...,150,195.0
...,...,...,...
98,เปา วินวอชลิควิดไวท์ฟลอรั่ลถุงเติม น้ำยาซักผ้า...,99,NaN
99,แอทแทค น้ำยาซักผ้า ไลฟ์ลี่ บลูมมิ่ง 700 มล.,95,NaN
100,แอทแทค น้ำยาซักผ้า ลัคชูรี่ โกลว์ 600 มล.,95,NaN
101,เปา วินวอชลิควิดเซ็นซวลไวโอเล็ต น้ำยาซักผ้า 62...,99,NaN


In [24]:
import re

# Define a function to extract info
def extract_product_info(name):
    # Extract Pack info inside parentheses
    pack_match = re.search(r'\((.*?)\)', name)
    pack = pack_match.group(1) if pack_match else None

    # Remove the pack part to avoid confusing the volume extraction
    clean_name = re.sub(r'\(.*?\)', '', name)

    # Extract Volume and Unit (Digits followed by units like มล. or ลิตร)
    # This looks for numbers potentially followed by decimals and then the unit
    vol_match = re.search(r'(\d+(?:\.\d+)?)\s*(มล\.|ลิตร|ก\.ก\.)', clean_name)
    volume = vol_match.group(1) if vol_match else None
    unit = vol_match.group(2) if vol_match else None

    return volume, unit, pack

# Apply the extraction
df[['Volume', 'Unit', 'Pack']] = df['Product Name'].apply(lambda x: pd.Series(extract_product_info(x)))

# Display the results
df[['Product Name', 'Volume', 'Unit', 'Pack']].head(15)

,Product Name,Volume,Unit,Pack
0,ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ (แกลลอ...,2800,มล.,แกลลอน
1,ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ 1250 มล.,1250,มล.,None
2,ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข้น 3in1 500 มล. ...,500,มล.,แพ็ก 3 ชิ้น
3,ไฟน์ไลน์ น้ำยาซักผ้า โปรคลีน ชมพู 550 มล. (1 แ...,550,มล.,1 แพ็ก 3 ชิ้น
4,ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข้น สำหรับซักกลาง...,550,มล.,แพ็ก 3 ชิ้น
5,ไฮยีน เอ็กซ์เพิร์ท วอช ผลิตภัณฑ์ซักผ้าชนิดน้ำ ...,600,มล.,None
6,ไฮยีน น้ำยาซักผ้า มิลค์กี้ ทัช 600 มล.,600,มล.,None
7,ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข้น ดีลักซ์เพอร์ฟ...,1250,มล.,None
8,ไฮยีน น้ำยาซักผ้า มิราเคิล 600 มล.,600,มล.,None
9,ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข้น เอ็กซ์ตร้า เฟ...,550,มล.,แพ็ก 3 ชิ้น


In [29]:
df.to_excel(f"7_eleven_laundry_{today_date}.xlsx")